In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('finviz_market_data_20260403_1438.csv')
print(f"Shape: {df.shape}")
print(df.dtypes)

Shape: (1000, 11)
No             int64
Ticker        object
Company       object
Sector        object
Industry      object
Country       object
Market Cap    object
P/E           object
Price         object
Change        object
Volume        object
dtype: object


In [3]:
def parse_market_cap(val):
    if val == '-' or pd.isna(val):
        return np.nan
    val = str(val).strip()
    for suffix, multiplier in [('T', 1e12), ('B', 1e9), ('M', 1e6), ('K', 1e3)]:
        if val.endswith(suffix):
            try:
                return float(val[:-1]) * multiplier
            except:
                return np.nan
    try:
        return float(val)
    except:
        return np.nan

df['MarketCap_Num']  = df['Market Cap'].apply(parse_market_cap)
df['PE_Num']         = pd.to_numeric(df['P/E'].replace('-', np.nan), errors='coerce')
df['Change_Num']     = pd.to_numeric(df['Change'].str.replace('%', ''), errors='coerce')
df['Volume_Num']     = pd.to_numeric(df['Volume'].str.replace(',', ''), errors='coerce')
df['Price']          = pd.to_numeric(df['Price'], errors='coerce')

# Cap category from market cap
def cap_category(v):
    if pd.isna(v):   return 'Unknown'
    if v >= 200e9:   return 'Mega Cap'
    if v >= 10e9:    return 'Large Cap'
    if v >= 2e9:     return 'Mid Cap'
    if v >= 300e6:   return 'Small Cap'
    return 'Micro Cap'

df['Cap_Category'] = df['MarketCap_Num'].apply(cap_category)

# Direction flags
df['Is_Gainer']  = df['Change_Num'] > 0
df['Is_Loser']   = df['Change_Num'] < 0
df['Abs_Change'] = df['Change_Num'].abs()

# Log-scaled features (handles skew)
df['Log_Volume']    = np.log1p(df['Volume_Num'])
df['Log_MarketCap'] = np.log1p(df['MarketCap_Num'])

# Price bands
df['Price_Band'] = pd.cut(
    df['Price'].astype(float),
    bins=[0, 5, 20, 50, 100, 200, 500, float('inf')],
    labels=['Penny', 'Very Low', 'Low', 'Mid', 'High', 'Very High', 'Ultra High']
)

In [4]:
print("\n=== Market Cap Categories ===")
print(df['Cap_Category'].value_counts())

print("\n=== Gainers / Losers / Flat ===")
print(f"Gainers: {df['Is_Gainer'].sum()}, Losers: {df['Is_Loser'].sum()}, Flat: {(df['Change_Num']==0).sum()}")

print("\n=== Price Bands ===")
print(df['Price_Band'].value_counts().sort_index())

print("\n=== Top 10 Gainers ===")
print(df.nlargest(10, 'Change_Num')[['Ticker','Company','Sector','Price','Change_Num','Cap_Category']].to_string())

print("\n=== Top 10 Losers ===")
print(df.nsmallest(10, 'Change_Num')[['Ticker','Company','Sector','Price','Change_Num','Cap_Category']].to_string())

print("\n=== Sector Stats ===")
sector_stats = df.groupby('Sector').agg(
    Count        = ('Ticker', 'count'),
    Avg_Change   = ('Change_Num', 'mean'),
    Gainers      = ('Is_Gainer', 'sum'),
    Losers       = ('Is_Loser', 'sum'),
    Avg_Price    = ('Price', 'mean'),
    Avg_PE       = ('PE_Num', 'mean'),
    Total_Volume = ('Volume_Num', 'sum')
).round(2)
sector_stats['Win_Rate_pct'] = (sector_stats['Gainers'] / sector_stats['Count'] * 100).round(1)
print(sector_stats.sort_values('Avg_Change', ascending=False).to_string())

print("\n=== Cap Category vs Avg Change ===")
print(df.groupby('Cap_Category')['Change_Num'].agg(['mean', 'std', 'count']).round(3))

print("\n=== Correlation Matrix ===")
numeric_cols = ['Price', 'Change_Num', 'Volume_Num', 'PE_Num', 'MarketCap_Num']
print(df[numeric_cols].corr().round(3))

print("\n=== Country Stats ===")
country_stats = df.groupby('Country').agg(
    Count      = ('Ticker', 'count'),
    Avg_Change = ('Change_Num', 'mean'),
    Gainers    = ('Is_Gainer', 'sum'),
).round(2)
country_stats['Win_Rate'] = (country_stats['Gainers'] / country_stats['Count'] * 100).round(1)
print(country_stats.sort_values('Count', ascending=False).head(15))



=== Market Cap Categories ===
Cap_Category
Unknown      357
Micro Cap    228
Small Cap    183
Mid Cap      130
Large Cap     91
Mega Cap      11
Name: count, dtype: int64

=== Gainers / Losers / Flat ===
Gainers: 529, Losers: 430, Flat: 36

=== Price Bands ===
Price_Band
Penny         186
Very Low      256
Low           334
Mid           112
High           68
Very High      35
Ultra High      4
Name: count, dtype: int64

=== Top 10 Gainers ===
    Ticker                                Company              Sector   Price  Change_Num Cap_Category
992   BDRX        Biodexa Pharmaceuticals Plc ADR          Healthcare    0.87       42.23    Micro Cap
19    AAOX           Tradr 2X Long AAOI Daily ETF           Financial   26.33       40.43      Unknown
307   AIXI                        Xiao-I Corp ADR          Technology    0.13       33.10    Micro Cap
705   ATPC                         Agape ATP Corp  Consumer Defensive    3.17       25.79    Micro Cap
672   ASTX           Tradr 2X Long A

In [5]:
# Z-score outliers on daily % change (|z| > 3)
mean_c = df['Change_Num'].mean()
std_c  = df['Change_Num'].std()
df['Change_ZScore'] = (df['Change_Num'] - mean_c) / std_c

print("\n=== Z-Score Outliers (|z| > 3) ===")
outliers = df[df['Change_ZScore'].abs() > 3][[
    'Ticker', 'Company', 'Sector', 'Change_Num', 'Change_ZScore', 'Cap_Category', 'Volume_Num'
]]
print(outliers.sort_values('Change_ZScore', ascending=False).to_string())

# IQR outliers on P/E
q1 = df['PE_Num'].quantile(0.25)
q3 = df['PE_Num'].quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
pe_outliers = df[df['PE_Num'] > upper_fence][['Ticker', 'Company', 'Sector', 'PE_Num', 'Price', 'Cap_Category']]
print(f"\n=== P/E IQR Outliers (>{upper_fence:.1f}) — {len(pe_outliers)} tickers ===")
print(pe_outliers.nlargest(10, 'PE_Num').to_string())



=== Z-Score Outliers (|z| > 3) ===
    Ticker                                  Company                  Sector  Change_Num  Change_ZScore Cap_Category  Volume_Num
992   BDRX          Biodexa Pharmaceuticals Plc ADR              Healthcare       42.23      10.457811    Micro Cap   120555421
19    AAOX             Tradr 2X Long AAOI Daily ETF               Financial       40.43      10.008585      Unknown     2732572
307   AIXI                          Xiao-I Corp ADR              Technology       33.10       8.179238    Micro Cap   218493016
705   ATPC                           Agape ATP Corp      Consumer Defensive       25.79       6.354881    Micro Cap      449730
672   ASTX             Tradr 2X Long ASTS Daily ETF               Financial       20.73       5.092057      Unknown     2900908
17    AAOI              Applied Optoelectronics Inc              Technology       20.34       4.994725      Mid Cap    19811717
186   AERT                    Aeries Technology Inc             Indu

In [6]:
ind = df.groupby('Industry').agg(
    Count      = ('Ticker', 'count'),
    Avg_Change = ('Change_Num', 'mean'),
    Win_Rate   = ('Is_Gainer', 'mean'),
    Avg_Volume = ('Volume_Num', 'mean')
).round(3)
ind['Win_Rate_pct'] = (ind['Win_Rate'] * 100).round(1)
ind_filtered = ind[ind['Count'] >= 5].sort_values('Avg_Change', ascending=False)

print("\n=== Top 20 Industries by Avg Change (min 5 tickers) ===")
print(ind_filtered.head(20).to_string())

print("\n=== Bottom 10 Industries ===")
print(ind_filtered.tail(10).to_string())


=== Top 20 Industries by Avg Change (min 5 tickers) ===
                                          Count  Avg_Change  Win_Rate    Avg_Volume  Win_Rate_pct
Industry                                                                                         
Communication Equipment                       9       4.274     0.778  4.506782e+06          77.8
Oil & Gas E&P                                 5       4.114     0.800  1.365166e+07          80.0
Entertainment                                 8       3.445     0.875  5.293017e+06          87.5
Medical Care Facilities                      10       2.737     0.600  8.629541e+05          60.0
Capital Markets                               7       2.487     0.571  1.330833e+06          57.1
Semiconductor Equipment & Materials          10       2.068     0.600  2.976869e+06          60.0
Semiconductors                               11       1.828     0.545  6.778429e+06          54.5
Information Technology Services               7       1.754  

In [7]:
print("\n=== Median P/E by Sector ===")
pe_sector = df[df['PE_Num'].notna()].groupby('Sector')['PE_Num'].agg(['mean', 'median', 'count']).round(2)
print(pe_sector.sort_values('median', ascending=False))

print("\n=== Losing Stocks with High P/E (PE > 50) — potential value traps ===")
risky = df[(df['PE_Num'] > 50) & df['Is_Loser']][[
    'Ticker', 'Company', 'Sector', 'PE_Num', 'Change_Num', 'Price', 'Cap_Category'
]]
print(risky.sort_values('PE_Num', ascending=False).head(15).to_string())



=== Median P/E by Sector ===
                          mean  median  count
Sector                                       
Technology               80.38   37.92     42
Healthcare               43.70   25.24     31
Basic Materials          36.27   24.40     16
Industrials             176.56   23.32     51
Utilities                20.99   21.96     10
Energy                   38.89   18.96      7
Consumer Cyclical        25.12   18.17     30
Consumer Defensive       84.31   18.05     11
Real Estate             187.61   17.97     18
Financial                45.66   12.84    100
Communication Services   12.20    9.29     11

=== Losing Stocks with High P/E (PE > 50) — potential value traps ===
    Ticker                                 Company              Sector   PE_Num  Change_Num   Price Cap_Category
473   ANPA               Rich Sparkle Holdings Ltd         Industrials  6884.62       -4.48    8.95    Micro Cap
113    ACR            ACRES Commercial Realty Corp         Real Estate  277

In [8]:
print("\n=== Top 10 Tickers by Volume ===")
print(df.nlargest(10, 'Volume_Num')[['Ticker', 'Company', 'Sector', 'Volume_Num', 'Change_Num', 'Cap_Category']].to_string())

print("\n=== Avg Volume by Sector ===")
print(df.groupby('Sector')['Volume_Num'].mean().sort_values(ascending=False).apply(lambda x: f"{x/1e6:.2f}M"))



=== Top 10 Tickers by Volume ===
    Ticker                          Company                  Sector  Volume_Num  Change_Num Cap_Category
307   AIXI                  Xiao-I Corp ADR              Technology   218493016       33.10    Micro Cap
992   BDRX  Biodexa Pharmaceuticals Plc ADR              Healthcare   120555421       42.23    Micro Cap
13     AAL      American Airlines Group Inc             Industrials    52701537       -2.61      Mid Cap
881   BATL               Battalion Oil Corp                  Energy    43512831        9.35    Micro Cap
46    ABEV                   Ambev S.A. ADR      Consumer Defensive    42957245       -1.35    Large Cap
394    AMC   AMC Entertainment Holdings Inc  Communication Services    40939094        8.74    Small Cap
398    AMD       Advanced Micro Devices Inc              Technology    38260180        3.47     Mega Cap
447   AMZN                   Amazon.com Inc       Consumer Cyclical    31255136       -0.38     Mega Cap
24    AAPL           

In [9]:
print("\n=== Penny Stocks (<$5) ===")
penny = df[df['Price'] < 5]
print(f"Count: {len(penny)}, Gainers: {penny['Is_Gainer'].sum()}, Losers: {penny['Is_Loser'].sum()}")
print(f"Avg Change: {penny['Change_Num'].mean():.2f}%, Std Dev: {penny['Change_Num'].std():.2f}%")

print("\n=== High-Priced Stocks (>$100) ===")
highp = df[df['Price'] > 100]
print(f"Count: {len(highp)}, Gainers: {highp['Is_Gainer'].sum()}, Losers: {highp['Is_Loser'].sum()}")
print(f"Avg Change: {highp['Change_Num'].mean():.2f}%, Std Dev: {highp['Change_Num'].std():.2f}%")



=== Penny Stocks (<$5) ===
Count: 185, Gainers: 90, Losers: 86
Avg Change: 0.93%, Std Dev: 6.90%

=== High-Priced Stocks (>$100) ===
Count: 107, Gainers: 54, Losers: 53
Avg Change: 0.02%, Std Dev: 2.84%


In [10]:
df.to_csv('finviz_enriched.csv', index=False)
print("\nEnriched CSV saved to finviz_enriched.csv")
print(f"New columns added: {[c for c in df.columns if c not in ['No','Ticker','Company','Sector','Industry','Country','Market Cap','P/E','Price','Change','Volume']]}")



Enriched CSV saved to finviz_enriched.csv
New columns added: ['MarketCap_Num', 'PE_Num', 'Change_Num', 'Volume_Num', 'Cap_Category', 'Is_Gainer', 'Is_Loser', 'Abs_Change', 'Log_Volume', 'Log_MarketCap', 'Price_Band', 'Change_ZScore']
